# Qwen TTS — unified Weave Model

Single `QwenTTSModel(Model)` class exposing two tasks: `.design(...)` and `.clone(...)`. Both are independently traced `@weave.op` methods sharing the same model loader + audio helpers.

In [1]:
import os
import datetime
import wave
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import soundfile as sf
import torch
import weave
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download
from IPython.display import Audio
from pydantic import PrivateAttr
from qwen_tts import Qwen3TTSModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from weave import Model

load_dotenv(override=True)

/bin/sh: sox: command not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


True

In [2]:
HF_TOKEN = os.environ.get("HUGGINGFACEHUB_API_TOKEN")
login(token=HF_TOKEN)

weave.init(f"{os.getenv('WANDB_ACCOUNT')}/{os.getenv('WEAVE_PROJECT')}")

weave: wandb version 0.27.0 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: weave version 0.52.41 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jbutt.
weave: View Weave data at https://wandb.ai/wandb-smle/jb_qwen_tts/weave


## Module-level helpers

`_drop_audio_array` / `_open_input_audio_for_trace` are wired into the `@weave.op` decorators on the class methods. `transcribe_audio_from_file` is a module-level op that becomes a nested child of `clone` in the trace.

In [3]:
def _drop_audio_array(output):
    """Hide audio_array from the Weave trace; runtime return value is unchanged."""
    if isinstance(output, dict) and "audio_array" in output:
        return {k: v for k, v in output.items() if k != "audio_array"}
    return output


def _open_input_audio_for_trace(inputs):
    """Replace the input_audio string path with an opened wave file so Weave
    renders it as a playable audio widget in the trace (clone task only)."""
    path = inputs.get("input_audio")
    if isinstance(path, str):
        try:
            return {**inputs, "input_audio": wave.open(path, "rb")}
        except Exception:
            return inputs
    return inputs


@weave.op(
    call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}"
)
def transcribe_audio_from_file(filename: str, model_name: str = "openai/whisper-small") -> str:
    """Local HuggingFace Whisper transcription."""
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    model.config.forced_decoder_ids = None

    audio, sr = librosa.load(filename, sr=16000)
    input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features

    predicted_ids = model.generate(language="en", input_features=input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    return transcription[0]

## `QwenTTSModel` class

In [4]:
class QwenTTSModel(Model):
    model_size: str = "1.7B"

    # Lazy-loaded weight holders. Two separate ones because the two heads are
    # different ~3GB artifacts. Calling only .design or only .clone keeps the
    # other holder at None and never loads it.
    _design_model: Any = PrivateAttr(default=None)
    _clone_model: Any = PrivateAttr(default=None)

    # ── shared helpers ─────────────────────────────────────────────────────

    def _device(self):
        return torch.device("cuda") if torch.cuda.is_available() else torch.device("mps")

    def _get_model_path(self, model_type: str) -> str:
        return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{self.model_size}-{model_type}")

    def _ensure_design_loaded(self):
        if self._design_model is None:
            print(f"Loading Qwen3-TTS-12Hz-{self.model_size}-VoiceDesign (first call only)...")
            self._design_model = Qwen3TTSModel.from_pretrained(
                self._get_model_path("VoiceDesign"),
                device_map=self._device(),
                dtype=torch.bfloat16,
                token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        return self._design_model

    def _ensure_clone_loaded(self):
        if self._clone_model is None:
            print(f"Loading Qwen3-TTS-12Hz-{self.model_size}-Base (first call only)...")
            self._clone_model = Qwen3TTSModel.from_pretrained(
                self._get_model_path("Base"),
                device_map=self._device(),
                dtype=torch.bfloat16,
                token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        return self._clone_model

    def _normalize_audio(self, x, eps=1e-12, clip=True):
        """Normalize audio to float32 in [-1, 1] range."""
        if np.issubdtype(x.dtype, np.integer):
            info = np.iinfo(x.dtype)
            if info.min < 0:
                y = x.astype(np.float32) / max(abs(info.min), info.max)
            else:
                mid = (info.max + 1) / 2.0
                y = (x.astype(np.float32) - mid) / mid
        elif np.issubdtype(x.dtype, np.floating):
            y = x.astype(np.float32)
            m = np.max(np.abs(y)) if y.size else 0.0
            if m > 1.0 + 1e-6:
                y = y / (m + eps)
        else:
            raise TypeError(f"Unsupported dtype: {x.dtype}")

        if clip:
            y = np.clip(y, -1.0, 1.0)
        if y.ndim > 1:
            y = np.mean(y, axis=-1).astype(np.float32)
        return y

    def _write_sound_to_file(self, output_filename, audio_file, sample_rate):
        sf.write(output_filename, audio_file, sample_rate, subtype="PCM_16")

    # ── voice design entry point ──────────────────────────────────────────

    @weave.op(
        call_display_name="VoiceDesign",
        postprocess_output=_drop_audio_array,
    )
    def design(self, text: str, language: str, voice_description: str):
        if not text or not text.strip():
            return None, "Error: Text is required."
        if not voice_description or not voice_description.strip():
            return None, "Error: Voice description is required."

        try:
            wavs, sr = self._ensure_design_loaded().generate_voice_design(
                text=text.strip(),
                language=language,
                instruct=voice_description.strip(),
                non_streaming_mode=True,
                max_new_tokens=2048,
            )
            out_dir = Path("./audio/designed_audio/")
            out_dir.mkdir(parents=True, exist_ok=True)
            output_filename = out_dir / "generated_audio.wav"
            self._write_sound_to_file(output_filename, wavs[0], sr)
            print(f"Voice design written to {output_filename}")
        except Exception as e:
            return None, f"Error: {type(e).__name__}: {e}"

        return {
            "audio": wave.open(str(output_filename), "rb"),
            "sample_rate": sr,
            "audio_array": wavs[0],
        }

    # ── voice clone entry point ───────────────────────────────────────────

    @weave.op(
        call_display_name="VoiceClone",
        postprocess_inputs=_open_input_audio_for_trace,
        postprocess_output=_drop_audio_array,
    )
    def clone(self, input_audio: str, target_text: str, max_new_tokens: int = 2048):
        # Nested @weave.op call — transcription appears as a child in the trace tree.
        ref_text = transcribe_audio_from_file(input_audio).strip()

        audio_data, sample_rate = librosa.load(input_audio, sr=None)
        ref_audio = (self._normalize_audio(audio_data), sample_rate)

        if not target_text or not target_text.strip():
            return None, "Error: Target text is required."
        if not ref_text or not ref_text.strip():
            return None, "Error: Reference text is required."

        try:
            wavs, sr = self._ensure_clone_loaded().generate_voice_clone(
                text=target_text.strip(),
                language="Auto",
                ref_audio=ref_audio,
                ref_text=ref_text,
                x_vector_only_mode=False,
                max_new_tokens=max_new_tokens,
            )
            out_dir = Path("./audio/cloned_audio/")
            out_dir.mkdir(parents=True, exist_ok=True)
            output_filename = out_dir / "generated_cloned_audio.wav"
            self._write_sound_to_file(output_filename, wavs[0], sr)
            print(f"Voice clone written to {output_filename}")
        except Exception as e:
            return None, f"Error: {type(e).__name__}: {e}"

        return {
            "generated_audio": wave.open(str(output_filename), "rb"),
            "sample_rate": sr,
            "audio_array": wavs[0],
        }

## Instantiate

In [5]:
qwen_tts = QwenTTSModel(model_size="1.7B")

## Voice design demo

First call loads the `VoiceDesign` 1.7B weights (slow). Subsequent calls reuse them.

In [6]:
design_result = qwen_tts.design(
    text="You wouldn't suspect little old me, would you?",
    language="Auto",
    voice_description=(
        "Use a playful, seductive and mature female voice. "
        "The vocal signature should be female but deeper."
    ),
)

Loading Qwen3-TTS-12Hz-1.7B-VoiceDesign (first call only)...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

weave: 🍩 https://wandb.ai/wandb-smle/jb_qwen_tts/r/call/019e7956-961b-78fc-a90b-74dc3f4f18a2
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Voice design written to audio/designed_audio/generated_audio.wav


In [ ]:
display(
    Audio(
        design_result["audio_array"],
        rate=design_result["sample_rate"],
        autoplay=False,
    )
)

## Voice clone demo

First call loads the `Base` 1.7B weights (slow). The transcription happens inside `clone` via the nested `transcribe_audio_from_file` op (local Whisper).

In [ ]:
clone_result = qwen_tts.clone(
    input_audio="audio/input/batman_original.wav",
    target_text=(
        "Batman here... I use Weights and Biases to help me watch over Gotham City. "
        "Whether it's the Joker, the penguin or Two-Face, Weights and Biases has you covered."
    ),
    max_new_tokens=2048,
)

In [ ]:
display(
    Audio(
        clone_result["audio_array"],
        rate=clone_result["sample_rate"],
        autoplay=False,
    )
)